# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [1]:
## Environment setup

# install piper-sample-generator (currently only supports linux systems)
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad

# install openwakeword (full installation to support training)
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword
!cd openwakeword

# install other dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==3.6.0
!pip install deep-phonemizer==0.0.19

# Download required models (workaround for Colab)
import os
os.makedirs("./openwakeword/openwakeword/resources/models")
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite


Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 184 (delta 70), reused 50 (delta 50), pack-reused 97 (from 1)
Receiving objects: 100% (184/184), 1.04 MiB | 20.52 MiB/s, done.
Resolving deltas: 100% (94/94), done.
--2026-11-13 08:36:01--  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
Resolving github.com (github.com)... 57.155.223.240
Connecting to github.com (github.com)|57.155.223.240|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/642029941/73f4af3c-7cf8-4547-a7b9-3bd29e7f3c33?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-11-13T09%3A26%3A35Z&rscd=attachment%3B+filename%3Den_US-libritts_r-medium.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-9

In [2]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [3]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

README.md:   0%|          | 0.00/936 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

270it [03:57,  1.14it/s]


In [8]:
## Download noise and background audio
# (replaced: the original AudioSet .tar and FMA streaming downloads no longer work)

import os, glob, random, subprocess
import pyarrow as pa, pyarrow.parquet as pq
from tqdm import tqdm

def to_wav16k(src, dst):
    subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", src,
                    "-ar", "16000", "-ac", "1", dst])

# ---------- AudioSet: one shard (~690 MB), converted to 16 kHz ----------
os.makedirs("audioset_16k", exist_ok=True)
if not os.path.exists("audioset_09.parquet"):
    !wget -q --show-progress -O audioset_09.parquet https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train/09.parquet
pf = pq.ParquetFile("audioset_09.parquet")
audio_col = [f.name for f in pf.schema_arrow
             if pa.types.is_struct(f.type) and f.type.get_field_index("bytes") >= 0][0]
i = 0
with tqdm(total=pf.metadata.num_rows, desc="AudioSet") as bar:
    for batch in pf.iter_batches(columns=[audio_col], batch_size=32):
        for item in batch.column(0).to_pylist():
            bar.update(1)
            if not item or not item.get("bytes"):
                continue
            with open("tmp_audio", "wb") as fh:
                fh.write(item["bytes"])
            to_wav16k("tmp_audio", f"audioset_16k/audioset_{i}.wav")
            i += 1
!rm -f tmp_audio audioset_09.parquet

# ---------- FMA music: official zip (7.2 GB), converted to 16 kHz ----------
n_hours = 5   # 120 clips x 30 s = 1 hour of music
os.makedirs("fma", exist_ok=True)
!wget -q --show-progress -O fma_small.zip https://os.unil.cloud.switch.ch/fma/fma_small.zip
!unzip -q -o fma_small.zip -d fma_raw
mp3s = sorted(glob.glob("fma_raw/**/*.mp3", recursive=True))
random.seed(0); random.shuffle(mp3s)
for i, f in enumerate(tqdm(mp3s[: n_hours * 120], desc="FMA")):
    to_wav16k(f, f"fma/fma_{i}.wav")
!rm -rf fma_raw fma_small.zip

print(len(os.listdir("audioset_16k")), "AudioSet files |", len(os.listdir("fma")), "FMA files")

AudioSet: 100%|██████████| 500/500 [01:13<00:00,  6.78it/s]


fma_small.zip       100%[===================>]   7.15G  22.1MB/s    in 5m 37s  


FMA: 100%|██████████| 600/600 [02:27<00:00,  4.06it/s]


500 AudioSet files | 600 FMA files


In [9]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

--2026-11-13 09:20:54--  https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
Resolving huggingface.co (huggingface.co)... 13.35.202.34, 13.35.202.40, 13.35.202.121, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.34|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/64f3a0b6918ffcc15af6923c/7e1cade4c3fda6a5081158383c8d43c4a3e1e42555150b596b373efddf9b5194?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27openwakeword_features_ACAV100M_2000_hrs_16bit.npy%3B+filename%3D%22openwakeword_features_ACAV100M_2000_hrs_16bit.npy%22%3B&user_id=public&X-Xet-Cas-Uid=public&xip=SyGQEZosKiI&Expires=1794565254&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRmM2EwYjY5MThmZmNjMTVhZjY5MjNjLzdlMWNhZGU0YzNmZGE2YTUwODExNTgzODNjOGQ0M2M0YTNlMWU0MjU1NTE1MGI1OTZiMzczZWZkZGY5YjUxOTRcXD9yZXNwb25zZS1

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [10]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

{'model_name': 'my_model',
 'target_phrase': ['hey jarvis'],
 'custom_negative_phrases': [],
 'n_samples': 10000,
 'n_samples_val': 2000,
 'tts_batch_size': 50,
 'augmentation_batch_size': 16,
 'piper_sample_generator_path': './piper-sample-generator',
 'output_dir': './my_custom_model',
 'rir_paths': ['./mit_rirs'],
 'background_paths': ['./background_clips'],
 'background_paths_duplication_rate': [1],
 'false_positive_validation_data_path': './validation_set_features.npy',
 'augmentation_rounds': 1,
 'feature_data_files': {'ACAV100M_sample': './openwakeword_features_ACAV100M_2000_hrs_16bit.npy'},
 'batch_n_per_class': {'ACAV100M_sample': 1024,
  'adversarial_negative': 50,
  'positive': 50},
 'model_type': 'dnn',
 'layer_size': 32,
 'steps': 50000,
 'max_negative_weight': 1500,
 'target_false_positives_per_hour': 0.2}

In [11]:
# Modify values in the config and save a new version

config["target_phrase"] = ["hey botty", "hey bot-ee"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 20000
config["n_samples_val"] = 2000
config["steps"] = 50000
config["target_accuracy"] = 0.7
config["target_recall"] = 0.5

# ADD: phrases that sound close to "hey botty" and must NOT trigger it
config["custom_negative_phrases"] = ["hey bobby", "hey buddy", "hey potty", "hey dotty", "hey betty","hey naughty"]

# ADD: how many false wake-ups per hour training should aim for
config["target_false_positives_per_hour"] = 0.2

config["background_paths"] = ['./audioset_16k', './fma']  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [13]:
# Install openWakeWord without its tflite-runtime requirement (no wheel for Python 3.13)
!pip install -q --no-deps -e ./openwakeword
!pip install -q onnxruntime scikit-learn requests

# Check it worked
!{sys.executable} -c "import openwakeword; print('openwakeword OK:', openwakeword.__file__)"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for openwakeword (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openwakeword 0.6.0 requires ai-edge-litert<3,>=2.0.2; platform_system == "Linux" or platform_system == "Darwin", which is not installed.
openwakeword 0.6.0 requires speexdsp-ns<1,>=0.1.2; platform_system == "Linux", which is not installed.
openwakeword OK: None


In [14]:
!pip install -q ai-edge-litert

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openwakeword 0.6.0 requires speexdsp-ns<1,>=0.1.2; platform_system == "Linux", which is not installed.


In [15]:
!cd /tmp && {sys.executable} -c "import openwakeword, os; print('openwakeword OK:', os.path.dirname(openwakeword.__file__))"

openwakeword OK: /content/openwakeword/openwakeword


In [17]:
# Disable the removed torchaudio.set_audio_backend() call in torch-audiomentations 0.11.0
p = "/usr/local/lib/python3.13/dist-packages/torch_audiomentations/utils/io.py"
s = open(p).read()
s = s.replace('torchaudio.set_audio_backend("soundfile")',
              'pass  # set_audio_backend was removed in torchaudio 2.x')
open(p, "w").write(s)

import torch, torchaudio
print("patched | torch", torch.__version__, "| torchaudio", torchaudio.__version__)

patched | torch 2.14.1+cu130 | torchaudio 2.11.0+cu130


In [19]:
# 1) Piper TTS runtime that the new piper-sample-generator uses
!pip install -q piper-tts==1.3.0

# 2) Bridge file so train.py's "from generate_samples import generate_samples" works again
shim = '''import os
from piper_sample_generator.__main__ import generate_samples as _generate_samples

_MODEL = os.path.join(os.path.dirname(os.path.abspath(__file__)),
                      "models", "en_US-libritts_r-medium.pt")

def generate_samples(*args, **kwargs):
    kwargs.setdefault("model", _MODEL)
    return _generate_samples(*args, **kwargs)
'''
open("piper-sample-generator/generate_samples.py", "w").write(shim)

# 3) Check the voice model and its config are both there
!ls -la piper-sample-generator/models/ | grep libritts_r

# 4) Quick test: generate 4 clips and play them (this is your pronunciation check)
!cd piper-sample-generator && {sys.executable} -c "from generate_samples import gples(text=['hey botty','hey bot-ee'], output_dir='/content/tts_test',max_samples=4, batch_size=2)"

import glob
from IPython.display import Audio, display
for f in sorted(glob.glob("/content/tts_test/*.wav")):
    print(f); display(Audio(f))

-rw-r--r-- 1 root root 204089915 Oct  6  2023 en_US-libritts_r-medium.pt
-rwxr-xr-x 1 root root     25782 Nov 13 08:36 en_US-libritts_r-medium.pt.json
  File "<string>", line 1
    from generate_samples import gples(text=['hey botty','hey bot-ee'], output_dir='/content/tts_test',max_samples=4, batch_size=2)
                                      ^
SyntaxError: invalid syntax


In [20]:
# Pronunciation test: generate 4 clips and play them
test = '''
from generate_samples import generate_samples
generate_samples(
    text=["hey botty", "hey bot-ee"],
    output_dir="/content/tts_test",
    max_samples=4,
    batch_size=2,
)
'''
open("piper-sample-generator/tts_test.py", "w").write(test)
!cd piper-sample-generator && {sys.executable} tts_test.py

import glob
from IPython.display import Audio, display
for f in sorted(glob.glob("/content/tts_test/*.wav")):
    print(f); display(Audio(f))

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
DEBUG:piper_sample_generator.__main__:Loading /content/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:piper_sample_generator.__main__:Successfully loaded the model
DEBUG:piper_sample_generator.__main__:CUDA available, using GPU
DEBUG:piper_sample_generator.__main__:Batch 1/2 complete
DEBUG:piper_sample_generator.__main__:Batch 2/2 complete
INFO:piper_sample_generator.__main__:Done
/content/tts_test/0.wav


/content/tts_test/1.wav


/content/tts_test/2.wav


/content/tts_test/3.wav


In [22]:
import glob, os

# 1) Find and remove the broken download (it's an error page, not a model)
found = glob.glob("/content/**/en_us_cmudict_forward.pt", recursive=True)
print("found:", found)
target = found[0] if found else "/content/openwakeword/openwakeword/resources/en_us_cmudict_forward.pt"
os.makedirs(os.path.dirname(target), exist_ok=True)
if os.path.exists(target):
    os.remove(target)

# 2) Download the real model from the Hugging Face mirror
!wget -q --show-progress -O "{target}" "https://huggingface.co/spaces/mostafaashahin/Phone-aid/resolve/main/models/d_phonemizer/en_us_cmudict_forward.pt"
print(os.path.getsize(target) / 1e6, "MB  (should be about 66.7)")

# 3) Let newer PyTorch load this older-format model file
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

found: ['/content/openwakeword/openwakeword/resources/en_us_cmudict_forward.pt']
/content/openwakewo 100%[===================>]  63.63M  26.1MB/s    in 2.4s    
66.725366 MB  (should be about 66.7)


In [23]:
# Step 1: Generate synthetic clips
# For the number of clips we are using, this should take ~10 minutes on a free Google Colab instance with a T4 GPU
# If generation fails, you can simply run this command again as it will continue generating until the
# number of files meets the targets specified in the config file

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

/usr/local/lib/python3.13/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
INFO:root:##################################################
Generating positive clips for training
##################################################
INFO:root:##################################################
Generating positive clips for testing
##################################################
INFO:root:##################################################
Generating negative clips for training
##################################################
/usr/local

In [25]:
import glob, numpy as np, soundfile as sf
from math import gcd
from scipy.signal import resample_poly
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

files = glob.glob("my_custom_model/hey_botty/*/*.wav")
print(len(files), "clips | example sample rate:", sf.info(files[0]).samplerate, "Hz")

def to_16k(f):
    data, sr = sf.read(f, dtype="float32")
    if sr == 16000:
        return 0
    g = gcd(sr, 16000)
    data = resample_poly(data, 16000 // g, sr // g)
    sf.write(f, np.clip(data, -1, 1), 16000, subtype="PCM_16")
    return 1

with ProcessPoolExecutor() as ex:
    changed = sum(tqdm(ex.map(to_16k, files, chunksize=64), total=len(files)))
print(changed, "clips converted to 16 kHz | check:", sf.info(files[0]).samplerate, "Hz")

44000 clips | example sample rate: 22050 Hz


100%|██████████| 44000/44000 [03:28<00:00, 211.18it/s]

44000 clips converted to 16 kHz | check: 16000 Hz


In [27]:
# Remove the half-written feature files from the failed Step 2 run
!ls -la my_custom_model/hey_botty/*.npy
!rm -f my_custom_model/hey_botty/*.npy
!ls my_custom_model/hey_botty/

-rw-r--r-- 1 root root 122880128 Nov 13 10:16 my_custom_model/hey_botty/positive_features_train.npy
negative_test  negative_train  positive_test  positive_train


In [29]:
# 1) Give torch-audiomentations replacements for torchaudio.info / torchaudio.load (removed in torchaudio 2.11)
p = "/usr/local/lib/python3.13/dist-packages/torch_audiomentations/utils/io.py"
s = open(p).read()
shim = '''import torchaudio
# --- CASABOT compat shim: torchaudio>=2.9 removed info()/old load(); use soundfile ---
import soundfile as _sf, torch as _torch
class _SfInfo:
    def __init__(self, f):
        i = _sf.info(str(f))
        self.num_frames, self.sample_rate, self.num_channels = i.frames, i.samplerate, i.channels
def _sf_load(f, frame_offset=0, num_frames=-1, normalize=True, channels_first=True, *a, **k):
    stop = None if (num_frames is None or num_frames < 0) else frame_offset + num_frames
    data, sr = _sf.read(str(f), start=frame_offset, stop=stop, dtype="float32", always_2d=True)
    t = _torch.from_numpy(data.T.copy() if channels_first else data.copy())
    return t, sr
torchaudio.info = lambda f, *a, **k: _SfInfo(f)
torchaudio.load = _sf_load
# --- end shim ---
'''
if "CASABOT compat shim" not in s:
    s = s.replace("import torchaudio\n", shim, 1)
    open(p, "w").write(s)
print("shim present:", "CASABOT compat shim" in open(p).read())

# 2) Remove the half-written feature files from this failed run
!rm -f my_custom_model/hey_botty/*.npy
!ls my_custom_model/hey_botty/

shim present: True
negative_test  negative_train  positive_test  positive_train


In [30]:
# Step 2: Augment the generated clips

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

/usr/local/lib/python3.13/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
INFO:root:##################################################
Computing openwakeword features for generated samples
##################################################
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
/usr/local/lib/python3

In [32]:
# 1) Package the new PyTorch ONNX exporter needs
!pip install -q onnxscript onnx

# 2) Make train.py save a backup of the trained model BEFORE exporting
p = "openwakeword/openwakeword/train.py"
s = open(p).read()
old = 'oww.export_model(model=best_model,'
if "_best.pt" not in s:
    s = s.replace(old, 'torch.save(best_model, os.path.join(config["output_dir"], config["model_name"] + "_best.pt")); ' + old, 1)
    open(p, "w").write(s)
print("backup patch in place:", "_best.pt" in open(p).read())

backup patch in place: True


In [35]:
# Step 3: Train model

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

/usr/local/lib/python3.13/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
INFO:root:##################################################
Starting training sequence 1...
##################################################
Training: 100% 49999/50000 [19:51<00:00, 41.97it/s]
INFO:root:##################################################
Starting training sequence 2...
##################################################
INFO:root:Increasing weight on negative examples to reduce false positives...
Training: 100% 4999/5000.0 [04:21<00:00, 19.

In [34]:

import torch, onnx

p = "openwakeword/openwakeword/train.py"
s = open(p).read()

# 1) Fix my backup line: save only the weights (state_dict), which always works
s = s.replace('torch.save(best_model, ', 'torch.save(best_model.state_dict(), ')

# 2) Test the ONNX export on a tiny model with the same layer types (takes seconds)
class T(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.f = torch.nn.Flatten(); self.l = torch.nn.Linear(16*96, 32)
        self.n = torch.nn.LayerNorm(32); self.o = torch.nn.Linear(32, 1)
    def forward(self, x):
        return torch.sigmoid(self.o(torch.relu(self.n(self.l(self.f(x))))))

def try_export(**kw):
    try:
        torch.onnx.export(T().eval(), torch.rand(1, 16, 96), "/tmp/t.onnx", opset_version=13, **kw)
        return [o.version for o in onnx.load("/tmp/t.onnx").opset_import]
    except Exception as e:
        return f"FAILED: {type(e).__name__}: {e}"[:300]

legacy = try_export(dynamo=False)
print("classic exporter (opset 13):", legacy)

# 3) If the classic exporter works, make train.py use it (keeps opset 13, best for .tflite later)
if isinstance(legacy, list) and 13 in legacy and "dynamo=False" not in s:
    s = s.replace("opset_version=13)", "opset_version=13, dynamo=False)")
open(p, "w").write(s)

print("backup fixed:", "best_model.state_dict()" in s, "| classic exporter used:", "dynamo=False" in s)

/tmp/ipykernel_1044/3243832405.py:20: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(T().eval(), torch.rand(1, 16, 96), "/tmp/t.onnx", opset_version=13, **kw)


classic exporter (opset 13): [13]
backup fixed: True | classic exporter used: True


In [37]:
import onnx, glob, shutil, numpy as np, onnxruntime as ort

src = "my_custom_model/hey_botty.onnx"
m = onnx.load(src)
old = m.graph.input[0].name

# 1) Rename the input to a plain name ("input") on a copy
for n in m.graph.node:
    new_inputs = ["input" if i == old else i for i in n.input]
    del n.input[:]; n.input.extend(new_inputs)
m.graph.input[0].name = "input"
onnx.save(m, "my_custom_model/hey_botty_renamed.onnx")

# 2) Convert, keeping the input shape exactly (1, 16, 96)
!rm -rf my_custom_model/tflite_out2
!onnx2tf -i my_custom_model/hey_botty_renamed.onnx -o my_custom_model/tflite_out2 -kat input -n

out = glob.glob("my_custom_model/tflite_out2/*_float32.tflite")[0]
shutil.copy(out, "my_custom_model/hey_botty.tflite")

# 3) Verify against the original ONNX model
from ai_edge_litert.interpreter import Interpreter
x = np.random.rand(1, 16, 96).astype(np.float32)
onnx_out = ort.InferenceSession(src).run(None, {old: x})[0]
it = Interpreter(model_path="my_custom_model/hey_botty.tflite"); it.allocate_tensors()
d_in, d_out = it.get_input_details()[0], it.get_output_details()[0]
print("tflite input shape:", d_in["shape"])
it.set_tensor(d_in["index"], x); it.invoke()
print("onnx:", float(onnx_out.ravel()[0]), "| tflite:", float(it.get_tensor(d_out["index"]).ravel()[0]))

flatbuffer_direct lowering: 100% 25/25 [00:00<00:00, 7643.24it/s]
flatbuffer_direct post-lowering [6/6] topological sort: 100% 6/6 [00:00<00:00, 52.69it/s]
flatbuffer_direct write timing: stage=float32 mode=builder_direct total=0.012s serialize=0.012s (sanitize=0.000s build=0.003s pack=0.009s output=0.000s) write=0.000s size=0.20MB
flatbuffer_direct write timing: stage=float16 mode=builder_direct total=0.006s serialize=0.006s (sanitize=0.000s build=0.000s pack=0.005s output=0.000s) write=0.000s size=0.10MB
flatbuffer_direct export [3/3] write float16 tflite: 100% 3/3 [00:00<00:00, 79.07it/s]
tflite input shape: [ 1 16 96]
onnx: 0.004638254642486572 | tflite: 0.004638254176825285


In [ ]:
# Step 4 (Optional): On Google Colab, sometimes the .tflite model isn't saved correctly
# If so, run this cell to retry

# Manually save to tflite as this doesn't work right in colab
def convert_onnx_to_tflite(onnx_model_path, output_path):
    """Converts an ONNX version of an openwakeword model to the Tensorflow tflite format."""
    # imports
    import onnx
    import logging
    import tempfile
    from onnx_tf.backend import prepare
    import tensorflow as tf

    # Convert to tflite from onnx model
    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model"))
        tflite_model = converter.convert()

        logging.info(f"####\nSaving tflite mode to '{output_path}'")
        with open(output_path, 'wb') as f:
            f.write(tflite_model)

    return None

convert_onnx_to_tflite(f"my_custom_model/{config['model_name']}.onnx", f"my_custom_model/{config['model_name']}.tflite")


After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!